# Multi-Horizon LOB Training Pipeline

**Full pipeline with checkpoint/recovery support.**

Run cells top-to-bottom. If the session is interrupted, simply re-run from Cell 1 — the checkpoint system will automatically skip already-completed architectures and resume in-progress ones from the last saved epoch.

### Recovery guarantee
- After every epoch, model weights + training state are saved to Drive/disk.
- On restart, completed architectures are detected and skipped.
- In-progress architectures resume from the last completed epoch.
- All paths live in `WEIGHTS_DIR` and `RESULTS_DIR` (on Drive in Colab).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (Colab / local)
# Run once per session. Safe to re-run.
# ═══════════════════════════════════════════════════════════════════════════
import os
from pathlib import Path


def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


IN_COLAB = is_colab()

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/multi-horizon-ofi')
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR    = str(PROJECT_ROOT / 'data' / 'processed')
WEIGHTS_DIR = str(PROJECT_ROOT / 'model_weights')
RESULTS_DIR = str(PROJECT_ROOT / 'results')

Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"DATA_DIR not found: {DATA_DIR}. "
        "In Colab, ensure Drive is mounted and the repo is at /content/drive/MyDrive/multi-horizon-ofi"
    )

tickers = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
    and any(f.endswith('.parquet') for f in os.listdir(os.path.join(DATA_DIR, d)))
])

print(f"IN_COLAB    : {IN_COLAB}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"WEIGHTS_DIR : {WEIGHTS_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"Tickers     : {tickers}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 2 — Imports
# ═══════════════════════════════════════════════════════════════════════════
import gc
import glob
import hashlib
import json
import math
import os
import random
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}")
print("Imports OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3 — Label Generation
# ═══════════════════════════════════════════════════════════════════════════
"""
Multi-horizon fixed-threshold classification labels.
Labels: 0=DOWN, 1=STATIONARY, 2=UP
"""

DEFAULT_HORIZONS: List[int] = [10, 20, 50, 100]


def _mid_price_array(df: pd.DataFrame, dtype=np.float64) -> np.ndarray:
    bp = df["bid_price_1"].to_numpy(dtype=dtype, copy=False)
    ap = df["ask_price_1"].to_numpy(dtype=dtype, copy=False)
    return (ap + bp) * 0.5


def _smoothed_mid(m: np.ndarray, k: int) -> np.ndarray:
    n = len(m)
    out = np.full(n, np.nan, dtype=m.dtype)
    if k >= n:
        return out
    cumsum = np.cumsum(m, dtype=m.dtype)
    out[: n - k] = (cumsum[k:] - cumsum[:-k]) / k
    return out


def make_fixed_threshold_classification_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    alpha: float = 0.002,
    use_smoothing: bool = True,
    dtype=np.float64,
) -> pd.DataFrame:
    if horizons is None:
        horizons = DEFAULT_HORIZONS
    m = _mid_price_array(df, dtype)
    n = len(m)
    out = np.empty((n, len(horizons)), dtype=dtype)
    for i, k in enumerate(horizons):
        future_m = _smoothed_mid(m, k) if use_smoothing else np.full(n, np.nan, dtype=dtype)
        if not use_smoothing and k < n:
            future_m[: n - k] = m[k:]
        pct_change = (future_m - m) / m
        labels = np.full(n, np.nan, dtype=dtype)
        labels[pct_change > alpha] = 2
        labels[pct_change < -alpha] = 0
        mid_mask = (~np.isnan(pct_change)) & (np.isnan(labels))
        labels[mid_mask] = 1
        out[:, i] = labels
    cols = [f"label_{k}" for k in horizons]
    return pd.DataFrame(out, index=df.index, columns=cols)


print("Label functions ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Global Config
# Tune these before training.
# ═══════════════════════════════════════════════════════════════════════════

DEEP_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Raw 10-level LOB columns (40 features per event)
DEEP_RAW_LOB_10_COLS = [
    f"{side}_{field}_{lvl}"
    for lvl in range(1, 11)
    for side, field in (("ask", "price"), ("ask", "size"), ("bid", "price"), ("bid", "size"))
]

DEEP_CONFIG: Dict = {
    # Paths (set by Cell 1)
    "data_dir":    DATA_DIR,
    "weights_dir": WEIGHTS_DIR,
    "results_dir": RESULTS_DIR,

    # Tickers (None → auto-discover)
    "tickers": None,

    # Horizons
    "horizons": [10, 20, 50, 100],

    # Label threshold
    "alpha": 0.00005,

    # Sequence length for windowed input
    "seq_len": 100,

    # Train/eval file split
    "train_file_fraction": 0.8,

    # Subsampling (reduce to fit in Colab RAM)
    "base_subsample":             4,
    "high_pressure_subsample":    8,
    "critical_pressure_subsample": 16,
    "max_rows_per_day_train":  8000,
    "max_rows_per_day_eval":  10000,
    "max_files_per_ticker":       0,   # 0 = all files

    # DataLoader
    "batch_size":   256,
    "num_workers":  0,

    # Optimiser
    "lr":            3e-4,
    "weight_decay":  1e-4,
    "grad_clip":     1.0,
    "amp":           True,    # use AMP if CUDA available

    # Loss
    "loss_mode":       "cb_focal",  # "ce" or "cb_focal"
    "cb_beta":          0.999,
    "cb_min_w":         0.5,
    "cb_max_w":         3.0,
    "cb_eps":           1.0,
    "focal_gamma":      2.0,
    "label_smoothing":  0.02,

    # Training loop
    "seed": 42,

    # Architectures to train (order matters for sequential execution)
    "run_architectures": ["dilated_transformer", "cnn_inception_lstm", "seq2seq_attn"],

    # Result file suffix
    "result_suffix": "_stable_es",
}

# ─── Training schedule ────────────────────────────────────────────────────
DEEP_MAX_EPOCHS          = 10   # max epochs per architecture
DEEP_EARLY_STOP_PATIENCE = 3    # stop if no improvement for N epochs
DEEP_EARLY_STOP_MIN_DELTA = 1e-4

print(f"Device      : {DEEP_DEVICE}")
print(f"Architectures: {DEEP_CONFIG['run_architectures']}")
print(f"Max epochs  : {DEEP_MAX_EPOCHS}  patience={DEEP_EARLY_STOP_PATIENCE}")
print(f"Config OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 5 — Checkpoint / Recovery System
# ═══════════════════════════════════════════════════════════════════════════
"""
The checkpoint system saves training state after EVERY epoch so that if
Colab crashes or times out, the next run can resume exactly where it left off.

State persisted per architecture:
  - Model weights (best so far)
  - Current epoch weights (for mid-epoch restart)
  - Epoch history (losses, f1 scores)
  - Best score / no_improve counter
  - Completed flag

All checkpoint files live in WEIGHTS_DIR (Google Drive in Colab).
"""
import json


def _ckpt_path(arch: str, suffix: str = "_stable_es") -> str:
    """Path for the JSON checkpoint file."""
    return os.path.join(WEIGHTS_DIR, f"{arch}{suffix}_checkpoint.json")


def _ckpt_weights_path(arch: str, label: str = "current", suffix: str = "_stable_es") -> str:
    """Path for a .pt weights file (current or best)."""
    return os.path.join(WEIGHTS_DIR, f"{arch}{suffix}_{label}_weights.pt")


def save_checkpoint(
    arch: str,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    best_score: float,
    best_epoch: int,
    no_improve: int,
    epoch_history: list,
    best_metrics: dict,
    completed: bool = False,
    suffix: str = "_stable_es",
) -> None:
    """
    Persist model + training state after each epoch.
    Called inside the training loop — safe to interrupt during next epoch.
    """
    # 1. Save current model weights
    cur_w_path = _ckpt_weights_path(arch, "current", suffix)
    torch.save(model.state_dict(), cur_w_path)

    # 2. Save JSON metadata
    meta = {
        "arch":          arch,
        "epoch":         int(epoch),
        "best_score":    float(best_score),
        "best_epoch":    int(best_epoch),
        "no_improve":    int(no_improve),
        "completed":     bool(completed),
        "epoch_history": epoch_history,
        "best_metrics":  best_metrics,
        "timestamp":     pd.Timestamp.now().isoformat(),
    }
    ckpt_p = _ckpt_path(arch, suffix)
    # Atomic-ish write: write to temp then rename
    tmp_p = ckpt_p + ".tmp"
    with open(tmp_p, "w") as f:
        json.dump(meta, f, indent=2)
    os.replace(tmp_p, ckpt_p)


def save_best_weights(
    arch: str,
    model: nn.Module,
    suffix: str = "_stable_es",
) -> str:
    """Save the best model weights separately."""
    best_w_path = _ckpt_weights_path(arch, "best", suffix)
    torch.save(model.state_dict(), best_w_path)
    return best_w_path


def load_checkpoint(arch: str, suffix: str = "_stable_es") -> dict | None:
    """
    Load checkpoint metadata for `arch`. Returns None if no checkpoint exists.
    """
    ckpt_p = _ckpt_path(arch, suffix)
    if not os.path.exists(ckpt_p):
        return None
    try:
        with open(ckpt_p) as f:
            return json.load(f)
    except Exception as e:
        print(f"[ckpt] WARNING: could not read checkpoint for {arch}: {e}")
        return None


def is_arch_completed(arch: str, suffix: str = "_stable_es") -> bool:
    """Return True if this architecture finished training."""
    ckpt = load_checkpoint(arch, suffix)
    return ckpt is not None and bool(ckpt.get("completed", False))


def restore_weights_to_model(
    arch: str,
    model: nn.Module,
    label: str = "current",
    suffix: str = "_stable_es",
) -> bool:
    """
    Load saved weights into model (in-place). Returns True on success.
    label: "current" or "best"
    """
    path = _ckpt_weights_path(arch, label, suffix)
    if not os.path.exists(path):
        return False
    try:
        sd = torch.load(path, map_location="cpu")
        model.load_state_dict(sd)
        return True
    except Exception as e:
        print(f"[ckpt] WARNING: could not load {label} weights for {arch}: {e}")
        return False


def print_checkpoint_status(suffix: str = "_stable_es") -> None:
    """Print recovery status for all architectures in DEEP_CONFIG."""
    archs = DEEP_CONFIG.get("run_architectures", [])
    print("\n" + "═" * 60)
    print("  Checkpoint Status")
    print("═" * 60)
    for arch in archs:
        ckpt = load_checkpoint(arch, suffix)
        if ckpt is None:
            status = "NOT STARTED"
        elif ckpt.get("completed"):
            ep  = ckpt.get('epoch', '?')
            sc  = ckpt.get('best_score', 0.0)
            status = f"COMPLETED  epoch={ep}  best_f1={sc:.4f}"
        else:
            ep = ckpt.get('epoch', 0)
            sc = ckpt.get('best_score', 0.0)
            ni = ckpt.get('no_improve', 0)
            status = f"IN PROGRESS  last_epoch={ep}  best_f1={sc:.4f}  no_improve={ni}"
        print(f"  {arch:<30} {status}")
    print("═" * 60 + "\n")


print_checkpoint_status()
print("Checkpoint system ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 6 — Training Utilities
# ═══════════════════════════════════════════════════════════════════════════
import psutil


def deep_set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _avail_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


def _deep_cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _deep_choose_subsample(cfg: dict) -> int:
    free = _avail_ram_gb()
    if free < 2.0:
        return int(cfg.get("critical_pressure_subsample", 16))
    if free < 4.0:
        return int(cfg.get("high_pressure_subsample", 8))
    return int(cfg.get("base_subsample", 4))


def _deep_choose_max_rows(cfg: dict, is_train: bool) -> int:
    key = "max_rows_per_day_train" if is_train else "max_rows_per_day_eval"
    default = 8000 if is_train else 10000
    return int(max(1, cfg.get(key, default)))


def _deep_seed_from_path(path: str, base_seed: int) -> int:
    digest = hashlib.md5(path.encode("utf-8")).hexdigest()
    return (int(digest[:8], 16) + int(base_seed)) % (2 ** 32 - 1)


def _deep_resolve_tickers(cfg: dict) -> list:
    if cfg.get("tickers"):
        return list(cfg["tickers"])
    data_dir = cfg["data_dir"]
    return sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
        and glob.glob(os.path.join(data_dir, d, "*.parquet"))
    ])


def _deep_collect_files_by_ticker(
    data_dir: str,
    tickers: list,
    max_files: int = 0,
) -> dict:
    files_by_ticker: Dict[str, List[str]] = {}
    for ticker in tickers:
        files = sorted(glob.glob(os.path.join(data_dir, ticker, "*.parquet")))
        if max_files > 0:
            files = files[:max_files]
        if files:
            files_by_ticker[ticker] = files
    return files_by_ticker


def _deep_split_train_eval_files(
    files_by_ticker: dict,
    train_frac: float = 0.8,
) -> Tuple[List[Tuple[str, str]], List[Tuple[str, str]]]:
    train_files: List[Tuple[str, str]] = []
    eval_files:  List[Tuple[str, str]] = []
    for ticker, files in files_by_ticker.items():
        n = len(files)
        n_train = max(1, min(n - 1, int(np.floor(n * train_frac)))) if n > 1 else 1
        for i, p in enumerate(files):
            (train_files if i < n_train else eval_files).append((ticker, p))
    return train_files, eval_files


def _deep_update_confusion(confusion: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    mask = (y_true >= 0) & (y_true < 3) & (y_pred >= 0) & (y_pred < 3)
    yt = y_true[mask].astype(np.int64)
    yp = y_pred[mask].astype(np.int64)
    np.add.at(confusion, (yt, yp), 1)


def _deep_metrics_from_confusion(confusion: np.ndarray, h: int) -> dict:
    metrics: Dict[str, float] = {}
    n_cls = confusion.shape[0]
    total = float(confusion.sum())
    metrics[f"h{h}_accuracy"] = float(np.trace(confusion)) / max(total, 1.0)
    f1s = []
    for c in range(n_cls):
        tp = float(confusion[c, c])
        fp = float(confusion[:, c].sum()) - tp
        fn = float(confusion[c, :].sum()) - tp
        prec = tp / max(tp + fp, 1e-9)
        rec  = tp / max(tp + fn, 1e-9)
        f1   = 2.0 * prec * rec / max(prec + rec, 1e-9)
        f1s.append(f1)
        metrics[f"h{h}_f1_c{c}"] = f1
    metrics[f"h{h}_f1_macro"]    = float(np.mean(f1s))
    metrics[f"h{h}_f1_weighted"] = float(
        sum(f1s[c] * float(confusion[c, :].sum()) for c in range(n_cls)) / max(total, 1.0)
    )
    return metrics


def _mean_macro_f1(metrics: dict, horizons: list) -> float:
    vals = [float(metrics.get(f"h{h}_f1_macro", 0.0)) for h in horizons]
    return float(np.mean(vals)) if vals else 0.0


print("Utilities ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 7 — Dataset (Stable Day-Window)
# ═══════════════════════════════════════════════════════════════════════════
from dataclasses import dataclass


@dataclass
class DayStats:
    ticker: str
    file_name: str
    rows_kept: int
    step: int
    max_rows: int


class StableDayWindowDataset(Dataset):
    """Per-day z-score normalization to prevent NaN explosions."""

    def __init__(self, raw: np.ndarray, starts: np.ndarray, labels: np.ndarray, seq_len: int):
        raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
        self.raw    = raw
        self.starts = starts.astype(np.int64, copy=False)
        self.labels = labels.astype(np.int64, copy=False)
        self.seq_len = int(seq_len)
        feat_mean = raw.mean(axis=0, dtype=np.float64).astype(np.float32)
        feat_std  = raw.std(axis=0,  dtype=np.float64).astype(np.float32)
        feat_std  = np.where(feat_std < 1e-6, 1.0, feat_std).astype(np.float32)
        self.feat_mean = feat_mean
        self.feat_std  = feat_std

    def __len__(self) -> int:
        return int(self.starts.size)

    def __getitem__(self, idx: int):
        s = int(self.starts[idx])
        x = (self.raw[s: s + self.seq_len] - self.feat_mean) / self.feat_std
        x = np.clip(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0), -10.0, 10.0)
        y = self.labels[idx]
        return torch.from_numpy(x.astype(np.float32, copy=False)), torch.from_numpy(y)


def _deep_build_day_dataset(
    parquet_path: str,
    cfg: dict,
    is_train: bool,
) -> Tuple[StableDayWindowDataset | None, DayStats | None]:
    horizons = list(cfg["horizons"])
    seq_len  = int(cfg["seq_len"])
    alpha    = float(cfg["alpha"])
    step     = _deep_choose_subsample(cfg)
    max_rows = _deep_choose_max_rows(cfg, is_train=is_train)

    try:
        df = pd.read_parquet(parquet_path, columns=DEEP_RAW_LOB_10_COLS)
    except Exception as e:
        print(f"  [dataset] Failed to read {parquet_path}: {e}")
        return None, None

    raw    = np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=False))
    raw    = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    y_full = make_fixed_threshold_classification_labels(
        df, horizons=horizons, alpha=alpha, use_smoothing=True
    ).to_numpy(dtype=np.float32, copy=False)

    max_h     = int(max(horizons))
    valid_end = len(df) - max_h
    del df

    if valid_end <= seq_len:
        gc.collect()
        return None, None

    labels     = y_full[seq_len - 1: valid_end]
    valid_mask = ~np.isnan(labels).any(axis=1)
    starts     = np.flatnonzero(valid_mask).astype(np.int64, copy=False)
    if step > 1:
        starts = starts[::step]
    if max_rows > 0:
        starts = starts[:max_rows]
    if starts.size == 0:
        gc.collect()
        return None, None

    y_day      = labels[starts].astype(np.int64, copy=False)
    class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
    starts     = starts[class_mask]
    y_day      = y_day[class_mask]

    if starts.size == 0:
        gc.collect()
        return None, None

    ds = StableDayWindowDataset(raw, starts, y_day, seq_len)
    stats = DayStats(
        ticker=os.path.basename(os.path.dirname(parquet_path)),
        file_name=os.path.basename(parquet_path),
        rows_kept=int(starts.size),
        step=step,
        max_rows=max_rows,
    )
    del raw, y_full, labels, valid_mask, y_day, class_mask
    gc.collect()
    return ds, stats


def _deep_make_loader(dataset: Dataset, cfg: dict, is_train: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=int(cfg.get("batch_size", 256)),
        shuffle=is_train,
        num_workers=int(cfg.get("num_workers", 0)),
        pin_memory=(DEEP_DEVICE.type == "cuda"),
        drop_last=is_train,
    )


print("Dataset classes ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 8 — Loss Functions (Class-Balanced Focal Loss)
# ═══════════════════════════════════════════════════════════════════════════

def _deep_class_weights_cb(labels: np.ndarray, cfg: dict) -> List[np.ndarray]:
    """Effective-Number-of-Samples class weights (Cui et al. CVPR 2019)."""
    beta  = float(cfg.get("cb_beta", 0.999))
    min_w = float(cfg.get("cb_min_w", 0.5))
    max_w = float(cfg.get("cb_max_w", 3.0))
    eps   = float(cfg.get("cb_eps", 1.0))
    weights: List[np.ndarray] = []
    for i in range(labels.shape[1]):
        y      = labels[:, i]
        counts = np.bincount(y.astype(np.int64, copy=False), minlength=3).astype(np.float64) + eps
        eff    = 1.0 - np.power(beta, counts)
        w      = (1.0 - beta) / np.maximum(eff, 1e-12)
        w      = w / np.mean(w)
        w      = np.clip(w, min_w, max_w)
        weights.append(w.astype(np.float32))
    return weights


def _deep_focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weight: Optional[torch.Tensor],
    gamma: float = 2.0,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    ce   = F.cross_entropy(logits, targets, weight=weight, reduction="none", label_smoothing=label_smoothing)
    p_t  = torch.exp(-ce)
    loss = ((1.0 - p_t) ** gamma) * ce
    return loss.mean()


def _deep_multihorizon_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    cfg: dict,
    weight_tensors: Optional[List[torch.Tensor]] = None,
) -> torch.Tensor:
    loss_mode       = str(cfg.get("loss_mode", "ce"))
    focal_gamma     = float(cfg.get("focal_gamma", 2.0))
    label_smoothing = float(cfg.get("label_smoothing", 0.0))
    losses = []
    for i in range(logits.shape[1]):
        w = None if weight_tensors is None else weight_tensors[i]
        if loss_mode == "cb_focal":
            l_i = _deep_focal_loss(logits[:, i, :], targets[:, i], w, focal_gamma, label_smoothing)
        else:
            l_i = F.cross_entropy(logits[:, i, :], targets[:, i], weight=w, label_smoothing=label_smoothing)
        losses.append(l_i)
    return torch.stack(losses).mean()


print("Loss functions ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 9 — Model Definitions
# ═══════════════════════════════════════════════════════════════════════════

# ── 1. Dilated CNN + Masked Transformer ──────────────────────────────────

class CausalDilatedConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int, dropout: float = 0.1):
        super().__init__()
        self.pad  = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, dilation=dilation, padding=self.pad)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.res  = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.conv(x)
        if self.pad > 0:
            y = y[..., :-self.pad]
        y = self.drop(F.gelu(self.bn(y)))
        r = self.res(x)
        if r.shape[-1] != y.shape[-1]:
            r = r[..., -y.shape[-1]:]
        return y + r


class DilatedMaskedTransformer(nn.Module):
    def __init__(self, input_dim: int, horizon_count: int, num_classes: int = 3,
                 d_model: int = 96, n_heads: int = 4, n_layers: int = 2,
                 dropout: float = 0.15, max_len: int = 1024):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes   = num_classes
        self.input_proj    = nn.Conv1d(input_dim, d_model, kernel_size=1)
        self.tcn = nn.ModuleList([
            CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=1, dropout=dropout),
            CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=2, dropout=dropout),
            CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=4, dropout=dropout),
        ])
        self.pos_emb = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.shape
        z = self.input_proj(x.transpose(1, 2))
        for block in self.tcn:
            z = block(z)
        z = z.transpose(1, 2)
        pos = self.pos_emb[:, :seq_len] if self.pos_emb.size(1) >= seq_len else \
            F.interpolate(self.pos_emb.transpose(1, 2), size=seq_len, mode="linear", align_corners=False).transpose(1, 2)
        z += pos
        attn_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=z.device), diagonal=1)
        z = self.norm(self.encoder(z, mask=attn_mask)[:, -1, :])
        return self.head(z).view(bsz, self.horizon_count, self.num_classes)


# ── 2. Hybrid CNN + Inception + LSTM ─────────────────────────────────────

class InceptionBlock1D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        b1 = out_ch // 4; b2 = out_ch // 4; b3 = out_ch // 4; b4 = out_ch - (b1 + b2 + b3)
        self.branch1 = nn.Sequential(nn.Conv1d(in_ch, b1, 1), nn.BatchNorm1d(b1), nn.GELU())
        self.branch2 = nn.Sequential(nn.Conv1d(in_ch, b2, 1), nn.BatchNorm1d(b2), nn.GELU(),
                                     nn.Conv1d(b2, b2, 3, padding=1), nn.BatchNorm1d(b2), nn.GELU())
        self.branch3 = nn.Sequential(nn.Conv1d(in_ch, b3, 1), nn.BatchNorm1d(b3), nn.GELU(),
                                     nn.Conv1d(b3, b3, 5, padding=2), nn.BatchNorm1d(b3), nn.GELU())
        self.branch4 = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                     nn.Conv1d(in_ch, b4, 1), nn.BatchNorm1d(b4), nn.GELU())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)


class HybridCNNInceptionLSTM(nn.Module):
    def __init__(self, input_dim: int, horizon_count: int, num_classes: int = 3,
                 channels: int = 96, lstm_hidden: int = 128, lstm_layers: int = 2, dropout: float = 0.15):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes   = num_classes
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(channels), nn.GELU(), nn.Dropout(dropout),
        )
        self.inception = InceptionBlock1D(channels, channels)
        self.pool      = nn.AdaptiveAvgPool1d(32)
        self.lstm      = nn.LSTM(channels, lstm_hidden, num_layers=lstm_layers, batch_first=True,
                                 dropout=dropout if lstm_layers > 1 else 0.0, bidirectional=False)
        self.drop      = nn.Dropout(dropout)
        self.head      = nn.Linear(lstm_hidden, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)
        z = self.stem(x.transpose(1, 2))
        z = self.inception(z)
        z = self.pool(z).transpose(1, 2)
        _, (h, _) = self.lstm(z)
        z = self.drop(h[-1])
        return self.head(z).view(bsz, self.horizon_count, self.num_classes)


# ── 3. Seq2Seq + Attention (DeepLOB-style) ───────────────────────────────

class Seq2SeqAttentionModel(nn.Module):
    def __init__(self, input_dim: int, horizon_count: int, num_classes: int = 3,
                 enc_hidden: int = 64, dec_hidden: int = 64, enc_layers: int = 2,
                 attn_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes   = num_classes
        self.encoder = nn.LSTM(input_dim, enc_hidden, num_layers=enc_layers, batch_first=True,
                               dropout=dropout if enc_layers > 1 else 0.0, bidirectional=True)
        enc_out_dim = enc_hidden * 2
        self.attn_w  = nn.Linear(enc_out_dim, attn_dim, bias=False)
        self.attn_v  = nn.Linear(attn_dim, 1, bias=False)
        self.decoder = nn.LSTM(enc_out_dim, dec_hidden, num_layers=1, batch_first=True)
        self.drop    = nn.Dropout(dropout)
        self.head    = nn.Linear(dec_hidden, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)
        enc_out, _ = self.encoder(x)                       # (B, T, 2*H)
        scores     = self.attn_v(torch.tanh(self.attn_w(enc_out))).squeeze(-1)
        weights    = torch.softmax(scores, dim=1).unsqueeze(-1)
        context    = (enc_out * weights).sum(dim=1, keepdim=True)  # (B, 1, 2*H)
        _, (h, _)  = self.decoder(context)
        z          = self.drop(h[-1])
        return self.head(z).view(bsz, self.horizon_count, self.num_classes)


# ── Model factory ─────────────────────────────────────────────────────────

def build_deep_model(arch: str, input_dim: int, horizon_count: int, num_classes: int = 3) -> nn.Module:
    if arch == "dilated_transformer":
        return DilatedMaskedTransformer(input_dim, horizon_count, num_classes)
    elif arch == "cnn_inception_lstm":
        return HybridCNNInceptionLSTM(input_dim, horizon_count, num_classes)
    elif arch == "seq2seq_attn":
        return Seq2SeqAttentionModel(input_dim, horizon_count, num_classes)
    else:
        raise ValueError(f"Unknown architecture: '{arch}'. "
                         f"Choose from: dilated_transformer, cnn_inception_lstm, seq2seq_attn")


# Sanity check
with torch.no_grad():
    for _arch in DEEP_CONFIG["run_architectures"]:
        _m = build_deep_model(_arch, len(DEEP_RAW_LOB_10_COLS), len(DEEP_CONFIG["horizons"]))
        _x = torch.randn(2, DEEP_CONFIG["seq_len"], len(DEEP_RAW_LOB_10_COLS))
        _y = _m(_x)
        assert _y.shape == (2, len(DEEP_CONFIG["horizons"]), 3), f"{_arch}: unexpected shape {_y.shape}"
        n_params = sum(p.numel() for p in _m.parameters())
        print(f"  {_arch:<30} params={n_params:,}  output={tuple(_y.shape)}")
        del _m, _x, _y

print("Models OK.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 10 — Core Training & Evaluation Loops
# ═══════════════════════════════════════════════════════════════════════════

def _train_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: "torch.cuda.amp.GradScaler",
    cfg: dict,
    train_files: List[Tuple[str, str]],
    horizons: List[int],
    epoch_idx: int,
    total_epochs: int,
    amp_enabled: bool,
    arch: str,
) -> dict:
    model.train()
    grad_clip       = float(cfg.get("grad_clip", 1.0))
    label_smoothing = float(cfg.get("label_smoothing", 0.0))

    day_losses: List[float] = []
    bad_batches  = 0
    fitted_days  = 0
    train_rows   = {h: 0 for h in horizons}
    train_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    for file_idx, (ticker, path) in enumerate(train_files, start=1):
        ds, stats = _deep_build_day_dataset(path, cfg, is_train=True)
        if ds is None:
            continue

        # Compute class weights from this day's labels
        weight_tensors = None
        if cfg.get("loss_mode", "ce") == "cb_focal":
            cb_ws = _deep_class_weights_cb(ds.labels, cfg)
            weight_tensors = [torch.tensor(w, dtype=torch.float32, device=DEEP_DEVICE) for w in cb_ws]

        loader = _deep_make_loader(ds, cfg, is_train=True)
        file_losses: List[float] = []

        for xb, yb in loader:
            xb = xb.to(DEEP_DEVICE, non_blocking=True)
            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)
            yb = yb.to(DEEP_DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                logits = model(xb)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                loss   = _deep_multihorizon_loss(logits, yb, cfg, weight_tensors)

            if not torch.isfinite(loss):
                bad_batches += 1
                del xb, yb, logits, loss
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()

            file_losses.append(loss.item())
            batch_n = int(yb.shape[0])
            for i, h in enumerate(horizons):
                train_rows[h]   += batch_n
                train_counts[h] += np.bincount(yb[:, i].detach().cpu().numpy(), minlength=3)

            del xb, yb, logits, loss

        if file_losses:
            day_losses.append(float(np.mean(file_losses)))
            fitted_days += 1

        print(
            f"[{arch}][ep {epoch_idx}/{total_epochs}][train {file_idx}/{len(train_files)}] "
            f"{ticker}/{stats.file_name}  rows={stats.rows_kept}  sub={stats.step}  "
            f"avg_loss={day_losses[-1]:.4f}  ram={_avail_ram_gb():.1f}GB"
        )

        del loader, ds
        _deep_cleanup_cuda()

    return {
        "avg_train_loss":   float(np.mean(day_losses)) if day_losses else float("nan"),
        "fitted_days":       int(fitted_days),
        "bad_batches":       int(bad_batches),
        "train_rows_seen":   train_rows,
        "train_class_counts": train_counts,
    }


def _eval_epoch(
    model: nn.Module,
    cfg: dict,
    eval_files: List[Tuple[str, str]],
    horizons: List[int],
    arch: str,
) -> dict:
    confusion    = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    eval_rows    = {h: 0 for h in horizons}
    eval_counts  = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    model.eval()
    with torch.no_grad():
        for file_idx, (ticker, path) in enumerate(eval_files, start=1):
            ds, stats = _deep_build_day_dataset(path, cfg, is_train=False)
            if ds is None:
                continue

            loader = _deep_make_loader(ds, cfg, is_train=False)
            for xb, yb in loader:
                xb    = torch.nan_to_num(xb.to(DEEP_DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)
                logits = torch.nan_to_num(model(xb), nan=0.0, posinf=0.0, neginf=0.0)
                preds  = logits.argmax(dim=2).detach().cpu().numpy().astype(np.int64)
                y_true = yb.numpy().astype(np.int64)
                for i, h in enumerate(horizons):
                    _deep_update_confusion(confusion[h], y_true[:, i], preds[:, i])
                    eval_rows[h]   += int(y_true.shape[0])
                    eval_counts[h] += np.bincount(y_true[:, i], minlength=3)
                del xb, logits, preds, y_true

            print(f"[{arch}][eval {file_idx}/{len(eval_files)}] {ticker}/{stats.file_name}  rows={stats.rows_kept}")
            del loader, ds
            _deep_cleanup_cuda()

    metrics: Dict[str, float] = {}
    for h in horizons:
        metrics.update(_deep_metrics_from_confusion(confusion[h], h))

    return {"metrics": metrics, "confusion": confusion, "eval_rows_seen": eval_rows, "eval_class_counts": eval_counts}


print("Training/eval loops ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 11 — Per-Architecture Training with Full Checkpoint Recovery
# ═══════════════════════════════════════════════════════════════════════════

def train_one_architecture(
    arch: str,
    cfg: dict,
    max_epochs: int = 10,
    patience: int = 3,
    min_delta: float = 1e-4,
    force_restart: bool = False,
) -> dict:
    """
    Train one architecture with per-epoch checkpoint recovery.

    On restart:
    - If the architecture is already marked COMPLETED, return the saved result.
    - If partially trained, load the last saved weights + state and resume
      from the next epoch.
    - If force_restart=True, ignore any existing checkpoint and start fresh.
    """
    suffix   = str(cfg.get("result_suffix", "_stable_es"))
    horizons = list(cfg["horizons"])
    amp_enabled = bool(cfg.get("amp", False)) and DEEP_DEVICE.type == "cuda"

    # ── Recover: skip if already completed ──────────────────────────────
    if not force_restart and is_arch_completed(arch, suffix):
        ckpt = load_checkpoint(arch, suffix)
        print(f"[{arch}] Already COMPLETED (epoch={ckpt['epoch']}  best_f1={ckpt['best_score']:.4f}). Skipping.")
        # Load saved run_meta from results file
        results_path = os.path.join(cfg["results_dir"], f"{arch}_results_day_streaming{suffix}.json")
        if os.path.exists(results_path):
            with open(results_path) as f:
                return json.load(f)
        return {"architecture": arch, "completed": True, "skipped": True}

    # ── File discovery ───────────────────────────────────────────────────
    tickers_list = _deep_resolve_tickers(cfg)
    if not tickers_list:
        raise FileNotFoundError(f"No tickers found in {cfg['data_dir']}")

    files_by_ticker = _deep_collect_files_by_ticker(
        cfg["data_dir"], tickers_list, int(cfg.get("max_files_per_ticker", 0))
    )
    if not files_by_ticker:
        raise FileNotFoundError("No parquet files found for selected tickers.")

    train_files, eval_files = _deep_split_train_eval_files(
        files_by_ticker, float(cfg.get("train_file_fraction", 0.8))
    )

    print(f"\n{'═'*80}")
    print(f"  Training: {arch}")
    print(f"  Device={DEEP_DEVICE}  train_files={len(train_files)}  eval_files={len(eval_files)}")
    print(f"  max_epochs={max_epochs}  patience={patience}  amp={amp_enabled}")

    # ── Build model ──────────────────────────────────────────────────────
    model = build_deep_model(
        arch=arch,
        input_dim=len(DEEP_RAW_LOB_10_COLS),
        horizon_count=len(horizons),
        num_classes=3,
    ).to(DEEP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg.get("lr", 3e-4)),
        weight_decay=float(cfg.get("weight_decay", 1e-4)),
    )
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    # ── Recover: resume from checkpoint if available ─────────────────────
    start_epoch   = 1
    best_score    = -1.0
    best_epoch    = 0
    best_metrics: dict = {}
    no_improve    = 0
    epoch_history: list = []

    best_eval_rows:   dict = {h: 0 for h in horizons}
    best_eval_counts: dict = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    agg_train_rows:   dict = {h: 0 for h in horizons}
    agg_train_counts: dict = {h: np.zeros(3, dtype=np.int64) for h in horizons}
    total_bad_batches = 0
    total_train_days  = 0

    if not force_restart:
        ckpt = load_checkpoint(arch, suffix)
        if ckpt is not None:
            loaded = restore_weights_to_model(arch, model, label="current", suffix=suffix)
            if loaded:
                model.to(DEEP_DEVICE)
                start_epoch   = int(ckpt["epoch"]) + 1
                best_score    = float(ckpt["best_score"])
                best_epoch    = int(ckpt["best_epoch"])
                no_improve    = int(ckpt["no_improve"])
                epoch_history = list(ckpt.get("epoch_history", []))
                best_metrics  = dict(ckpt.get("best_metrics", {}))
                print(f"  ✓ Resumed from epoch {start_epoch - 1}  best_f1={best_score:.4f}  no_improve={no_improve}")
                if no_improve >= patience:
                    print(f"  Early stopping already triggered before restart. Returning saved best.")
                    # Already done — mark completed and return
                    save_checkpoint(arch, int(ckpt["epoch"]), model, optimizer,
                                    best_score, best_epoch, no_improve, epoch_history, best_metrics,
                                    completed=True, suffix=suffix)
                    return _finalize_arch(arch, cfg, model, best_metrics, best_score, best_epoch,
                                         epoch_history, train_files, eval_files, horizons,
                                         agg_train_rows, agg_train_counts, best_eval_rows, best_eval_counts,
                                         total_bad_batches, total_train_days, 0.0, max_epochs, patience, min_delta, suffix)
            else:
                print(f"  WARNING: Checkpoint found but weights missing for {arch}. Starting fresh.")

    if start_epoch > max_epochs:
        print(f"[{arch}] All {max_epochs} epochs already done. Marking complete.")
        save_checkpoint(arch, max_epochs, model, optimizer, best_score, best_epoch,
                        no_improve, epoch_history, best_metrics, completed=True, suffix=suffix)
        return _finalize_arch(arch, cfg, model, best_metrics, best_score, best_epoch,
                              epoch_history, train_files, eval_files, horizons,
                              agg_train_rows, agg_train_counts, best_eval_rows, best_eval_counts,
                              total_bad_batches, total_train_days, 0.0, max_epochs, patience, min_delta, suffix)

    # ── Training loop ────────────────────────────────────────────────────
    start_t = time.time()

    for epoch in range(start_epoch, max_epochs + 1):
        print(f"\n[{arch}] ── Epoch {epoch}/{max_epochs} ──────────────────")

        train_info = _train_epoch(
            model=model, optimizer=optimizer, scaler=scaler,
            cfg=cfg, train_files=train_files, horizons=horizons,
            epoch_idx=epoch, total_epochs=max_epochs, amp_enabled=amp_enabled, arch=arch,
        )

        eval_info = _eval_epoch(model=model, cfg=cfg, eval_files=eval_files, horizons=horizons, arch=arch)

        metrics = eval_info["metrics"]
        score   = _mean_macro_f1(metrics, horizons)
        improved = score > (best_score + min_delta)

        # Accumulate stats
        total_bad_batches += int(train_info["bad_batches"])
        total_train_days  += int(train_info["fitted_days"])
        for h in horizons:
            agg_train_rows[h]   += int(train_info["train_rows_seen"][h])
            agg_train_counts[h] += train_info["train_class_counts"][h]

        epoch_history.append({
            "epoch":          int(epoch),
            "mean_macro_f1":  float(score),
            "avg_train_loss": float(train_info["avg_train_loss"]),
            "bad_batches":    int(train_info["bad_batches"]),
            "improved":       bool(improved),
        })

        print(f"[{arch}] epoch={epoch}  mean_macro_f1={score:.6f}  best={best_score:.6f}  improved={improved}")

        if improved:
            best_score    = float(score)
            best_epoch    = int(epoch)
            best_metrics  = dict(metrics)
            best_eval_rows   = {h: int(eval_info["eval_rows_seen"][h]) for h in horizons}
            best_eval_counts = {h: eval_info["eval_class_counts"][h].copy() for h in horizons}
            save_best_weights(arch, model, suffix)
            no_improve = 0
        else:
            no_improve += 1

        # ── Save checkpoint after every epoch (key recovery point) ───────
        save_checkpoint(
            arch=arch, epoch=epoch, model=model, optimizer=optimizer,
            best_score=best_score, best_epoch=best_epoch, no_improve=no_improve,
            epoch_history=epoch_history, best_metrics=best_metrics,
            completed=False, suffix=suffix,
        )
        print(f"[{arch}] ✓ Checkpoint saved (epoch={epoch})")

        if no_improve >= patience:
            print(f"[{arch}] Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

        _deep_cleanup_cuda()

    # ── Restore best weights ─────────────────────────────────────────────
    restored = restore_weights_to_model(arch, model, label="best", suffix=suffix)
    if not restored:
        print(f"[{arch}] WARNING: best weights not found, using current weights.")

    elapsed = time.time() - start_t

    # ── Mark completed ───────────────────────────────────────────────────
    save_checkpoint(
        arch=arch, epoch=best_epoch, model=model, optimizer=optimizer,
        best_score=best_score, best_epoch=best_epoch, no_improve=no_improve,
        epoch_history=epoch_history, best_metrics=best_metrics,
        completed=True, suffix=suffix,
    )

    return _finalize_arch(
        arch=arch, cfg=cfg, model=model, best_metrics=best_metrics,
        best_score=best_score, best_epoch=best_epoch, epoch_history=epoch_history,
        train_files=train_files, eval_files=eval_files, horizons=horizons,
        agg_train_rows=agg_train_rows, agg_train_counts=agg_train_counts,
        best_eval_rows=best_eval_rows, best_eval_counts=best_eval_counts,
        total_bad_batches=total_bad_batches, total_train_days=total_train_days,
        elapsed=elapsed, max_epochs=max_epochs, patience=patience, min_delta=min_delta, suffix=suffix,
    )


def _finalize_arch(
    arch, cfg, model, best_metrics, best_score, best_epoch, epoch_history,
    train_files, eval_files, horizons,
    agg_train_rows, agg_train_counts, best_eval_rows, best_eval_counts,
    total_bad_batches, total_train_days, elapsed, max_epochs, patience, min_delta, suffix,
) -> dict:
    """Save final weights + JSON result, return run_meta dict."""
    os.makedirs(cfg["weights_dir"], exist_ok=True)
    os.makedirs(cfg["results_dir"], exist_ok=True)

    weights_path = os.path.join(cfg["weights_dir"], f"{arch}{suffix}_weights.pt")
    torch.save({
        "architecture":   arch,
        "state_dict":     model.state_dict(),
        "input_dim":      len(DEEP_RAW_LOB_10_COLS),
        "horizons":       horizons,
        "seq_len":        int(cfg["seq_len"]),
        "alpha":          float(cfg["alpha"]),
        "device_trained": str(DEEP_DEVICE),
        "best_epoch":     int(best_epoch),
        "best_macro_f1":  float(best_score),
    }, weights_path)

    run_meta = {
        "timestamp":     pd.Timestamp.now().isoformat(),
        "mode":          "day_by_day_streaming_deep_stable_earlystop",
        "architecture":  arch,
        "horizons":      horizons,
        "seq_len":       int(cfg["seq_len"]),
        "alpha":         float(cfg["alpha"]),
        "train_config": {
            "max_epochs":       int(max_epochs),
            "patience":         int(patience),
            "min_delta":        float(min_delta),
            "batch_size":       int(cfg.get("batch_size", 256)),
            "lr":               float(cfg.get("lr", 3e-4)),
            "weight_decay":     float(cfg.get("weight_decay", 1e-4)),
            "grad_clip":        float(cfg.get("grad_clip", 1.0)),
            "amp":              bool(cfg.get("amp", False)),
            "label_smoothing":  float(cfg.get("label_smoothing", 0.0)),
            "device":           str(DEEP_DEVICE),
        },
        "early_stopping": {
            "best_epoch":       int(best_epoch),
            "best_mean_macro_f1": float(best_score),
            "epochs_ran":       int(len(epoch_history)),
        },
        "epoch_history":   epoch_history,
        "files": {
            "train_files": len(train_files),
            "eval_files":  len(eval_files),
            "days_fitted": int(total_train_days),
        },
        "train_rows_seen":    {f"h{h}": int(agg_train_rows[h]) for h in horizons},
        "eval_rows_seen":     {f"h{h}": int(best_eval_rows[h]) for h in horizons},
        "train_class_counts": {f"h{h}": [int(x) for x in agg_train_counts[h].tolist()] for h in horizons},
        "eval_class_counts":  {f"h{h}": [int(x) for x in best_eval_counts[h].tolist()] for h in horizons},
        "bad_batches_total": int(total_bad_batches),
        "test_metrics":      best_metrics,
        "model_path":        weights_path,
        "runtime_seconds":   round(elapsed, 2),
    }

    results_path = os.path.join(cfg["results_dir"], f"{arch}_results_day_streaming{suffix}.json")
    with open(results_path, "w") as f:
        json.dump(run_meta, f, indent=2)

    print(f"[{arch}] ✓ Weights  → {weights_path}")
    print(f"[{arch}] ✓ Results  → {results_path}")

    del model
    _deep_cleanup_cuda()
    return run_meta


print("train_one_architecture() ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 12 — RUN ALL ARCHITECTURES
#
# This cell is the main entry point. Run it once per session.
# On restart, it automatically:
#   - Skips completed architectures
#   - Resumes in-progress architectures from the last saved epoch
#
# To force a fresh restart for a specific arch, call:
#   train_one_architecture("arch_name", DEEP_CONFIG, force_restart=True, ...)
# ═══════════════════════════════════════════════════════════════════════════

deep_set_seed(int(DEEP_CONFIG["seed"]))

# Show status before starting
print_checkpoint_status()

all_runs: Dict[str, dict] = {}
pipeline_start = time.time()

for _arch in DEEP_CONFIG["run_architectures"]:
    print(f"\n{'▶'*3} Starting: {_arch}")
    try:
        _run_meta = train_one_architecture(
            arch=_arch,
            cfg=DEEP_CONFIG,
            max_epochs=DEEP_MAX_EPOCHS,
            patience=DEEP_EARLY_STOP_PATIENCE,
            min_delta=DEEP_EARLY_STOP_MIN_DELTA,
            force_restart=False,  # ← Set True to ignore checkpoint and retrain
        )
        all_runs[_arch] = _run_meta
    except Exception as _e:
        print(f"\n[ERROR] {_arch} failed with: {_e}")
        import traceback
        traceback.print_exc()
        print(f"Continuing to next architecture...")
        _deep_cleanup_cuda()
        continue

pipeline_elapsed = time.time() - pipeline_start

# ── Summary ──────────────────────────────────────────────────────────────
print("\n" + "═" * 80)
print("  PIPELINE COMPLETE")
print("═" * 80)
print(f"  Total time: {pipeline_elapsed/60:.1f} min")
print()

for _arch, _meta in all_runs.items():
    _f1   = _mean_macro_f1(_meta.get("test_metrics", {}), list(DEEP_CONFIG["horizons"]))
    _ep   = _meta.get("early_stopping", {}).get("best_epoch", _meta.get("best_epoch", "?"))
    _skip = _meta.get("skipped", False)
    _tag  = " (loaded from checkpoint)" if _skip else ""
    print(f"  {_arch:<30} mean_macro_f1={_f1:.4f}  best_epoch={_ep}{_tag}")

print()
print_checkpoint_status()

# Save summary
_summary = {
    "timestamp":     pd.Timestamp.now().isoformat(),
    "architectures": list(DEEP_CONFIG["run_architectures"]),
    "horizons":      list(DEEP_CONFIG["horizons"]),
    "max_epochs":    DEEP_MAX_EPOCHS,
    "patience":      DEEP_EARLY_STOP_PATIENCE,
    "runtime_seconds": round(pipeline_elapsed, 2),
    "results": {
        arch: {
            "best_mean_macro_f1": _mean_macro_f1(m.get("test_metrics", {}), list(DEEP_CONFIG["horizons"])),
            "best_epoch": m.get("early_stopping", {}).get("best_epoch", m.get("best_epoch", None)),
        }
        for arch, m in all_runs.items()
    },
}
_summary_path = os.path.join(RESULTS_DIR, "pipeline_summary.json")
with open(_summary_path, "w") as _f:
    json.dump(_summary, _f, indent=2)
print(f"Summary saved → {_summary_path}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 13 — Results & Per-Horizon Metrics
# Safe to run after Cell 12 or after a restart (loads from saved JSON).
# ═══════════════════════════════════════════════════════════════════════════

suffix  = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
archs   = DEEP_CONFIG["run_architectures"]
horizons = list(DEEP_CONFIG["horizons"])

print("\n" + "═" * 88)
print("  Results Summary")
print("═" * 88)

col_w = 10
header = f"{'Architecture':<30}" + "".join(f"{'h'+str(h)+' acc':>{col_w}}" for h in horizons) \
       + "".join(f"{'h'+str(h)+' f1m':>{col_w}}" for h in horizons) + f"{'mean f1m':>{col_w}}"
print(header)
print("-" * len(header))

for arch in archs:
    results_path = os.path.join(RESULTS_DIR, f"{arch}_results_day_streaming{suffix}.json")
    if not os.path.exists(results_path):
        # Try loading from all_runs if in memory
        meta = all_runs.get(arch) if "all_runs" in dir() else None
        if meta is None:
            print(f"  {arch:<30} — no results found")
            continue
    else:
        with open(results_path) as f:
            meta = json.load(f)

    metrics = meta.get("test_metrics", {})
    accs = [metrics.get(f"h{h}_accuracy", float("nan")) for h in horizons]
    f1s  = [metrics.get(f"h{h}_f1_macro",  float("nan")) for h in horizons]
    mean_f1 = float(np.nanmean(f1s)) if f1s else float("nan")

    row = f"{arch:<30}"
    row += "".join(f"{v:>{col_w}.4f}" for v in accs)
    row += "".join(f"{v:>{col_w}.4f}" for v in f1s)
    row += f"{mean_f1:>{col_w}.4f}"
    print(row)

print("═" * 88)

# Per-horizon detail
print("\nPer-horizon breakdown:")
for arch in archs:
    results_path = os.path.join(RESULTS_DIR, f"{arch}_results_day_streaming{suffix}.json")
    if not os.path.exists(results_path):
        continue
    with open(results_path) as f:
        meta = json.load(f)
    metrics = meta.get("test_metrics", {})
    best_ep = meta.get("early_stopping", {}).get("best_epoch", "?")
    print(f"\n  {arch}  (best_epoch={best_ep})")
    for h in horizons:
        acc = metrics.get(f"h{h}_accuracy", float("nan"))
        f1m = metrics.get(f"h{h}_f1_macro",  float("nan"))
        f1w = metrics.get(f"h{h}_f1_weighted", float("nan"))
        print(f"    h={h:>4}: acc={acc:.4f}  f1_macro={f1m:.4f}  f1_weighted={f1w:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 14 — Epoch History & Training Curves
# ═══════════════════════════════════════════════════════════════════════════

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available — printing text curves only")

suffix = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
archs  = DEEP_CONFIG["run_architectures"]

for arch in archs:
    results_path = os.path.join(RESULTS_DIR, f"{arch}_results_day_streaming{suffix}.json")
    if not os.path.exists(results_path):
        print(f"  {arch}: no results file yet")
        continue
    with open(results_path) as f:
        meta = json.load(f)
    history = meta.get("epoch_history", [])
    if not history:
        print(f"  {arch}: no epoch history")
        continue

    epochs = [h["epoch"] for h in history]
    f1s    = [h["mean_macro_f1"] for h in history]
    losses = [h["avg_train_loss"] for h in history]

    print(f"\n  {arch} — epoch history:")
    for ep, f1, loss in zip(epochs, f1s, losses):
        marker = " ◀ best" if f1 == max(f1s) else ""
        print(f"    ep={ep:>3}  mean_macro_f1={f1:.4f}  avg_train_loss={loss:.4f}{marker}")

    if HAS_MPL:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
        ax1.plot(epochs, f1s, marker="o")
        ax1.set_title(f"{arch} — mean macro F1")
        ax1.set_xlabel("epoch"); ax1.set_ylabel("f1")
        ax2.plot(epochs, losses, marker="o", color="orange")
        ax2.set_title(f"{arch} — avg train loss")
        ax2.set_xlabel("epoch"); ax2.set_ylabel("loss")
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, f"{arch}_training_curve.png"), dpi=100)
        plt.show()
        print(f"  Saved training curve → {arch}_training_curve.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 15 — (OPTIONAL) Reset / Force-Restart a Specific Architecture
#
# Run this cell only if you want to ERASE the checkpoint for a specific
# architecture and start it fresh next time Cell 12 runs.
#
# Usage:
#   RESET_ARCH = "dilated_transformer"  # change to your target
#   Then uncomment and run the block below.
# ═══════════════════════════════════════════════════════════════════════════

# RESET_ARCH = "dilated_transformer"   # ← change this
# _suffix = str(DEEP_CONFIG.get("result_suffix", "_stable_es"))
#
# _paths_to_remove = [
#     _ckpt_path(RESET_ARCH, _suffix),
#     _ckpt_weights_path(RESET_ARCH, "current", _suffix),
#     _ckpt_weights_path(RESET_ARCH, "best",    _suffix),
# ]
# for _p in _paths_to_remove:
#     if os.path.exists(_p):
#         os.remove(_p)
#         print(f"Deleted: {_p}")
#     else:
#         print(f"Not found (skipped): {_p}")
# print(f"Reset complete for {RESET_ARCH}. Re-run Cell 12 to retrain.")

print("Cell 15 — reset utility (commented out by default). Uncomment to use.")